# <font color = 'red'> DEPENDENCIAS

In [3]:
import pandas as pd
import numpy as np
import plotly.express as px
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.preprocessing import MinMaxScaler

import sys
import os

# Agregar la carpeta calibration_code al path
sys.path.append(os.path.abspath("../../calibration_code"))

# Ahora puedes importar los módulos personalizados
from modelling_tools import (plot_histogram, plot_univariate_freq, assign_deciles, count_categories_by_decile, 
                             calculate_category_proportions, summarize_decile_analysis, summarize_grouped_deciles, group_deciles,
                             compute_odds_ratio)
from visualization_tools import plot_interactive_chart
from utils import g
from config import get_data_path, get_code_path
from data_cleaning import check_dataframe_quality

# <font color = 'red'> CARGA DE DATOS

In [4]:
df = pd.read_csv(get_data_path("bivariate_preprocessed_data.csv"))

In [5]:
res = check_dataframe_quality(df)

No missing values found.
No infinite values found.
No duplicate rows found.


# <font color = 'red'> ANÁLISIS

In [6]:
col = "Total_EMI_per_month"

## <font color = 'skyblue'> ANÁLISIS GENERAL

Las medianas tienen el orden esperado: Median Bad > Mediana Standard > Mediana Good.

Esto concuerda con la hipótesis de que los deudores con cuotas más altas tienden a pagar peor (mayor endeudamiento o menor liquidez).

In [7]:
fig_box = px.box(df, x="Credit_Mix", y=col, title=f"Distribution of {col} by Credit Score Category")
fig_box.show()

## <font color = 'skyblue'> ANÁLISIS POR DECILES

In [8]:
continuous_variable= col
decile_col_name = continuous_variable + '_Decile'
target_col_string = "Credit_Mix" # variable dependiente con nombres string
target_col = 'Credit_Score' # variable dependiente int (para modelos)

In [9]:
analysis_summary = summarize_decile_analysis(df, continuous_variable, decile_col_name, target_col_string)

# Obtener los resultados
df_deciles = analysis_summary["df_deciles"]  # DataFrame con los deciles asignados
deciles_summary = analysis_summary["decile_summary"]  # Resumen de deciles con conteos y proporciones
display(deciles_summary)
res = check_dataframe_quality(df_deciles)

,Decile_Min,Decile_Max,Decile_Count,Decile_Proportion,count_Bad,count_Good,count_Standard,prop_Bad,prop_Good,prop_Standard
Total_EMI_per_month_Decile,,,,,,,,,,
0,-1.000000,0.000000,13510,0.13510,966,6810,5734,0.071503,0.504071,0.424426
1,0.008317,18.944548,6493,0.06493,664,2046,3783,0.102264,0.315109,0.582627
2,18.945348,32.176647,10002,0.10002,1716,2889,5397,0.171566,0.288842,0.539592
3,32.186040,46.388945,9995,0.09995,2337,2633,5025,0.233817,0.263432,0.502751
4,46.390730,61.541296,10003,0.10003,2664,2471,4868,0.266320,0.247026,0.486654
5,61.589793,80.270575,10001,0.10001,3071,2555,4375,0.307069,0.255474,0.437456
6,80.271752,110.291282,10003,0.10003,2805,2802,4396,0.280416,0.280116,0.439468
7,110.331145,156.604085,10000,0.10000,2964,2794,4242,0.296400,0.279400,0.424200
8,156.653146,217.705293,9997,0.09997,2886,2816,4295,0.288687,0.281685,0.429629


No missing values found.
No infinite values found.
No duplicate rows found.


In [10]:
df[(df[continuous_variable] >= 35.665094) & (df[continuous_variable] <= 37.317021) & (df['Credit_Score'] == 0)].shape

(285, 92)

Proporción de Buenos: se observa una relación negativa, (en tendencia), entre la magnitud de las cuotas y la proporción de buenos deudores.

Proporción de standard: la realación entre la magnitud de las cuotas y la proporción de deudores Standard es mixta: una relación positiva hasta los 18 dólares, por encima de este umbral, la relación es negativa.

Proporción de malos: según lo esperado, la proporción de malos y la magnitud de las cuotas se relacionan positivamente.

In [13]:
chart_types = {
    "prop_Good":"line",
    "prop_Standard":"line",
    "prop_Bad": "line",  
    "Decile_Count": "bar"    
}

fig = plot_interactive_chart(
    df=deciles_summary,  
    y_columns=["prop_Bad", "prop_Good", "prop_Standard", "Decile_Count"],  
    x_column="Decile_Max",  
    chart_types=chart_types, 
    title=f"Proportion of Credit Score Categories by {continuous_variable}",
    x_title="Decile",
    y_title="Decile Count",  
    y2_title="Proportion",   
    secondary_y=["prop_Bad", "prop_Good", "prop_Standard"],  
    width=900,
    height=500,
    custom_colors={"prop_Bad": "red", "prop_Good":"lightgreen",
    "prop_Standard":"brown", "Decile_Count": "gray"}  
)

fig.show()

<font color = 'brown'> Agrupación de deciles

Se agrupan deciles buscando una relación monótona entre las proporciones y la variable:

In [14]:
group_map = {0: "Group_1", 
             1: "Group_2", 
             2: "Group_3", 
             3: "Group_4", 
             4: "Group_5",
             5: "Group_6", 
             6: "Group_6",
             7: "Group_6",
             8: "Group_6",
             9: "Group_7"}

# Agrupar los deciles
grouped_col_name = "Grouped_" + continuous_variable

df_deciles_grouped = group_deciles(df_deciles, decile_col_name, grouped_col_name, group_map)

summary_results = summarize_grouped_deciles(df_deciles_grouped, grouped_col_name, continuous_variable, target_col_string, prefix="Decile_")
grouped_deciles_summary = summary_results['df']


# Definir el mapeo manual de los grupos a enteros
group_mapping = {
    'Group_1': 1,
    'Group_2': 2,
    'Group_3': 3,
    'Group_4': 4,
    'Group_5': 5,
    'Group_6': 6,
    'Group_7': 7,
}

if not set(group_map.values()) == set(group_mapping.keys()):
    print("Problemas en el mapeo de grupos a enteros!")

# Usar `.map()` en lugar de `.replace()` para evitar el warning
df_deciles_grouped[grouped_col_name] = (
    df_deciles_grouped[grouped_col_name]
    .map(group_mapping)  # Mapear los valores
    .astype("Int64")      # Convertir a entero manejando NaN si existen
)

display(grouped_deciles_summary)

res = check_dataframe_quality(df_deciles_grouped)

,Decile_Min,Decile_Max,Decile_Count,Decile_Proportion,count_Bad,count_Good,count_Standard,prop_Bad,prop_Good,prop_Standard
Grouped_Total_EMI_per_month,,,,,,,,,,
Group_1,-1.000000,0.000000,13510,0.13510,966,6810,5734,0.071503,0.504071,0.424426
Group_2,0.008317,18.944548,6493,0.06493,664,2046,3783,0.102264,0.315109,0.582627
Group_3,18.945348,32.176647,10002,0.10002,1716,2889,5397,0.171566,0.288842,0.539592
Group_4,32.186040,46.388945,9995,0.09995,2337,2633,5025,0.233817,0.263432,0.502751
Group_5,46.390730,61.541296,10003,0.10003,2664,2471,4868,0.266320,0.247026,0.486654
Group_6,61.589793,217.705293,40001,0.40001,11726,10967,17308,0.293143,0.274168,0.432689
Group_7,217.734258,357.406068,9996,0.09996,3695,2568,3733,0.369648,0.256903,0.373449


No missing values found.
No infinite values found.
No duplicate rows found.


In [16]:
chart_types = {
    "prop_Good":"line",
    "prop_Standard":"line",
    "prop_Bad": "line",  
    "Decile_Count": "bar"    
}

fig = plot_interactive_chart(
    df=grouped_deciles_summary,  
    y_columns=["prop_Bad", "prop_Good", "prop_Standard", "Decile_Count"],  
    x_column="Decile_Max",  
    chart_types=chart_types, 
    title=f"Proportion of Credit Score Categories by {continuous_variable}",
    x_title="Decile",
    y_title="Decile Count",  
    y2_title="Proportion",   
    secondary_y=["prop_Bad", "prop_Good", "prop_Standard"],  
    width=900,
    height=500,
    custom_colors={"prop_Bad": "red", "prop_Good":"lightgreen",
    "prop_Standard":"brown", "Decile_Count": "gray"}  
)

fig.show()

## <font color = 'skyblue'> REGRESIONES BIVARIADAS

Como Credit_Score tiene tres categorías (Bad, Standard, Good) se pueden usar dos enfoques 
de regresión categórica: 

- Regresión Logística Multinomial → No asume orden en las categorías (como si fueran colores: rojo, azul, verde).
- Regresión Logística Ordinal → Asume que hay un orden en las categorías (Bad < Standard < Good).

Dado que hay un orden entre las categorías se utiliza Regresión Logística Ordinal:

<font color = 'gold'> Sin Agrupaciones

In [17]:
df_ = df_deciles_grouped.copy()
x_variable = continuous_variable
res = check_dataframe_quality(df_)

No missing values found.
No infinite values found.
No duplicate rows found.


In [18]:
# para no generar problemas numéricos debe escalarse esta variable.
# se opta por normalizar la variable:

g(df_[[x_variable]].describe()).transpose()

,count,mean,std,min,25%,50%,75%,max
Total_EMI_per_month,"100,000.00",88.21,83.24,-1.00,25.59,61.54,131.53,357.41


Todos los coeficientes son significativos.

Total_EMI_per_month_Scaled -1.2571: el coeficiente es negativo: por cada dólar adicional en el pago de la cuota, la probabilidad de estar en una categoría superior de Credit_Score disminuye.

Threshold 0/1 -1.4907: Umbral que separa las categorías Bad y Standard. Si la puntuación supera este umbral, es más probable que 
sea standard en lugar de bad.

Threshold 1/2 0.7093: Umbral que separa las categorías Standard y Good. Si la puntuación supera este umbral, es más probable que 
sea Good en lugar de Standard.

In [19]:
df_ = df_deciles_grouped.copy()
x_variable = continuous_variable

scaler = MinMaxScaler()
df_[continuous_variable + "_Scaled"] = scaler.fit_transform(df_[[x_variable]])

res = check_dataframe_quality(df_)

# Ajustar el modelo de regresión logística ordinal con la variable escalada
model_income = OrderedModel(df_[target_col], df_[continuous_variable + "_Scaled"], distr="logit")
result_income = model_income.fit(method='bfgs')

# Mostrar resumen del modelo
print(result_income.summary())

# Calcular e interpretar el Odds Ratio
res_odds = compute_odds_ratio(result_income, variable_name=continuous_variable + "_Scaled", description="Ingreso Anual Escalado")
print(res_odds["interpretation"])


No missing values found.
No infinite values found.
No duplicate rows found.
Optimization terminated successfully.
         Current function value: 1.049283
         Iterations: 10
         Function evaluations: 12
         Gradient evaluations: 12
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:            -1.0493e+05
Model:                   OrderedModel   AIC:                         2.099e+05
Method:            Maximum Likelihood   BIC:                         2.099e+05
Date:                Sun, 06 Apr 2025                                         
Time:                        13:28:58                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                                 coef    

<font color = 'gold'> Por Deciles

Todos los coeficientes son significativos.

Total_EMI_per_month_Decile -0.1210: Por cada decil adicional, la probabilidad de estar en una categoría superior de Credit_Score disminuye.

Threshold 0/1 -1.7261: Umbral que separa las categorías Bad y Standard. Si la puntuación supera este umbral, es más probable que 
sea standard en lugar de bad.

Threshold 1/2 0.7182: Umbral que separa las categorías Standard y Good. Si la puntuación supera este umbral, es más probable que 
sea Good en lugar de Standard.

In [20]:
df_ = df_deciles_grouped.copy()
x_variable = decile_col_name

res = check_dataframe_quality(df_)

model_age_decile = OrderedModel(df_[target_col], df_[x_variable], distr="logit")

result_age_decile = model_age_decile.fit(method='bfgs')

print(result_age_decile.summary())

res_odds = compute_odds_ratio(result_age_decile, variable_name = x_variable, description = 'Decil de Ingreso Neto')
print(res_odds['interpretation'])

No missing values found.
No infinite values found.
No duplicate rows found.
Optimization terminated successfully.
         Current function value: 1.043544
         Iterations: 10
         Function evaluations: 12
         Gradient evaluations: 12
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:            -1.0435e+05
Model:                   OrderedModel   AIC:                         2.087e+05
Method:            Maximum Likelihood   BIC:                         2.087e+05
Date:                Sun, 06 Apr 2025                                         
Time:                        13:32:28                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                                 coef    

<font color = 'gold'> Por Agrupamientos de Deciles

Todos los coeficientes son significativos.

Grouped_Total_EMI_per_month -0.2013: Por cada grupo adicional, la probabilidad de estar en una categoría superior de Credit_Score aumenta.

Threshold 0/1 -2.1119: Umbral que separa las categorías Bad y Standard.

Threshold 1/2 0.7249: Umbral que separa las categorías Standard y Good. Si la puntuación supera 0.7604, es más probable que 
sea Good en lugar de Standard.

In [21]:
df_ = df_deciles_grouped.copy()
x_variable = grouped_col_name

df_[target_col] = df_[target_col].astype(int)
df_[x_variable] = df_[x_variable].astype(int)

res = check_dataframe_quality(df_)

model_age_decile = OrderedModel(df_[target_col], df_[x_variable], distr="logit")

result_age_decile = model_age_decile.fit(method='bfgs')

print(result_age_decile.summary())

res_odds = compute_odds_ratio(result_age_decile, variable_name = x_variable, description = 'Decil de Edad')
print(res_odds['interpretation'])

No missing values found.
No infinite values found.
No duplicate rows found.
Optimization terminated successfully.
         Current function value: 1.039072
         Iterations: 11
         Function evaluations: 13
         Gradient evaluations: 13
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:            -1.0391e+05
Model:                   OrderedModel   AIC:                         2.078e+05
Method:            Maximum Likelihood   BIC:                         2.078e+05
Date:                Sun, 06 Apr 2025                                         
Time:                        13:33:22                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                                  coef   

## <font color = 'skyblue'> CONCLUSIONES

### 📊 Comparación de Representaciones de `Total_EMI_per_month` usando Regresión Ordinal

| Representación                         | Coeficiente principal | Indicadores de ajuste                                                                 | Interpretación                                                                 |
|----------------------------------------|------------------------|----------------------------------------------------------------------------------------|--------------------------------------------------------------------------------|
| `Total_EMI_per_month_Scaled`          | -1.2571               | **Log-Likelihood**: -104,928<br>**AIC**: 209,857<br>**BIC**: 209,878                   | 🔹 Mejor ajuste global.<br>🔹 Mayor EMI mensual se asocia con menor calidad crediticia. |
| `Total_EMI_per_month_Decile`          | -0.1210               | **Log-Likelihood**: -104,354<br>**AIC**: 208,709<br>**BIC**: 208,730                   | 🔹 Ajuste intermedio.<br>🔹 Discretización reduce información, pero mantiene relación negativa. |
| `Grouped_Total_EMI_per_month`         | -0.2013               | **Log-Likelihood**: -103,907<br>**AIC**: 207,816<br>**BIC**: 207,837                   | 🔹 Menor ajuste relativo.<br>🔹 Aún así, mantiene sentido económico claro: más EMI implica mayor riesgo. |


## <font color = 'skyblue'> EXPORTACIÓN DE DATOS CON VARIABLES ADICIONALES

In [32]:
# df_deciles_grouped.to_csv("../../calibration_data/preprocessed_data.csv", index=False)
